In [37]:
import pandas as pd 
import numpy as np

In [38]:
df = pd.read_csv('results.csv')
df.head()
df['date'] = pd.to_datetime(df['date'])

In [39]:
conditions = [
    df['home_score'] > df['away_score'],
    df['home_score'] < df['away_score'],
    df['home_score'] == df['away_score']
]

results = [
    'home_win',
    'away_win',
    'tie'
]

df['Target'] = np.select(conditions, results, default='Error')
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,Target
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,tie
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,home_win
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,home_win
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,tie
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,home_win


In [41]:
# old -> new
df = df.sort_values('date').reset_index(drop=True)

h2h_historic = {}

h2h_home_wins = []
h2h_away_wins = []
h2h_ties = []

for index, row in df.iterrows():
    home = row['home_team']
    away = row['away_team']
    target = row['Target']
    
    game = tuple(sorted([home, away]))
    
    # if first game
    if game not in h2h_historic:
        h2h_historic[game] = {home: 0, away: 0, 'tie': 0}
        
    h2h_home_wins.append(h2h_historic[game][home])
    h2h_away_wins.append(h2h_historic[game][away])
    h2h_ties.append(h2h_historic[game]['tie'])
    
    # updates
    if target == 'home_win':
        h2h_historic[game][home] += 1
    elif target == 'away_win':
        h2h_historic[game][away] += 1
    elif target == 'tie':
        h2h_historic[game]['tie'] += 1

# back to pandas
df['h2h_home_wins'] = h2h_home_wins
df['h2h_away_wins'] = h2h_away_wins
df['h2h_ties'] = h2h_ties

In [42]:
teams_historic = {}

home_scored_col = []
home_conceded_col = []
away_scored_col = []
away_conceded_col = []

for index, row in df.iterrows():
    home_score = row['home_score']
    away_score = row['away_score']
    game_date = row['date']

    home = row['home_team']
    away = row['away_team']
    if home not in teams_historic:
        teams_historic[home] = []
    if away not in teams_historic:
        teams_historic[away] = []

    eight_years_limit = game_date - pd.DateOffset(years=8)
    teams_historic[home] = [ game for game in teams_historic[home] if game['date'] > eight_years_limit]
    teams_historic[away] = [ game for game in teams_historic[away] if game['date'] > eight_years_limit]

    home_scored_goals = 0
    home_conceded_goals = 0
    for g in teams_historic[home]:
        home_scored_goals += g['scored']
        home_conceded_goals += g['conceded']

    away_scored_goals = 0
    away_conceded_goals = 0
    for g in teams_historic[away]:
        away_scored_goals += g['scored']
        away_conceded_goals += g['conceded']

    home_scored_col.append(home_scored_goals)
    home_conceded_col.append(home_conceded_goals)
    away_scored_col.append(away_scored_goals)
    away_conceded_col.append(away_conceded_goals)

    teams_historic[home].append({'date': game_date, 'scored': home_score, 'conceded': away_score})
    teams_historic[away].append({'date': game_date, 'scored': away_score, 'conceded': home_score})

df['home_scored'] = home_scored_col
df['home_conceded'] = home_conceded_col
df['away_scored'] = away_scored_col
df['away_conceded'] = away_conceded_col

    


In [43]:
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,Target,h2h_home_wins,h2h_away_wins,h2h_ties,home_scored,home_conceded,away_scored,away_conceded
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,tie,0,0,0,0.0,0.0,0.0,0.0
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,home_win,0,0,1,0.0,0.0,0.0,0.0
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,home_win,0,1,1,2.0,4.0,4.0,2.0
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,tie,1,1,1,5.0,4.0,4.0,5.0
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,home_win,1,1,2,6.0,7.0,7.0,6.0
